# Evaluate TorchXRayVision models in PadChest dataset using balanced subsets of data for each pathology 

Number of positive images to evaluate each pathology (We use the same number of positive/negative samples):

1. atelectasis: 3839
2. infiltration: 7503
3. pneumothorax: 217
4. effusion: 3246
5. pneumonia: 3307
6. cardiomegaly: 8114
7. nodule: 3540
8. mass: 783
9. fracture: 2524

# Imports

In [1]:
import os
import pandas as pd
import numpy as np
import torch
import tqdm
import sklearn
from sklearn import metrics
import torchvision
import torchxrayvision as xrv
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from typing import Dict, List
from skimage.io import imread
import random
from sklearn.metrics import confusion_matrix
import imblearn

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Device: {DEVICE}')

Device: cuda


# Load models

In [2]:
## Load 224x224 models 
model_pc = xrv.models.DenseNet(weights="densenet121-res224-pc")
model_rsna = xrv.models.DenseNet(weights="densenet121-res224-rsna") # RSNA Pneumonia Challenge
model_nih = xrv.models.DenseNet(weights="densenet121-res224-nih") # NIH chest X-ray8
model_chex = xrv.models.DenseNet(weights="densenet121-res224-chex") # CheXpert (Stanford)
model_mimic_nb = xrv.models.DenseNet(weights="densenet121-res224-mimic_nb") # MIMIC-CXR (MIT)
model_mimic_ch = xrv.models.DenseNet(weights="densenet121-res224-mimic_ch") # MIMIC-CXR (MIT)

# 512x512 models
model_all = xrv.models.ResNet(weights="resnet50-res512-all")

# Move the models to the specified device
model_pc.to(DEVICE)
model_rsna.to(DEVICE)
model_nih.to(DEVICE)
model_chex.to(DEVICE)
model_mimic_nb.to(DEVICE)
model_mimic_ch.to(DEVICE)
model_all.to(DEVICE)

XRV-ResNet-resnet50-res512-all

# Create PC dataset

In [3]:
class Dataset:
    """The datasets in this library aim to fit a simple interface where the
    imgpath and csvpath are specified. Some datasets require more than one
    metadata file and for some the metadata files are packaged in the library
    so only the imgpath needs to be specified.
    """
    def __init__(self):
        pass

    pathologies: List[str]

    labels: np.ndarray

    csv: pd.DataFrame

    def totals(self) -> Dict[str, Dict[str, int]]:
        """Compute counts of pathologies.

        Returns: A dict containing pathology name -> (label->value)
        """
        counts = [dict(collections.Counter(items[~np.isnan(items)]).most_common()) for items in self.labels.T]
        return dict(zip(self.pathologies, counts))

    def __repr__(self) -> str:
        """Returns the name and a description of the dataset such as:

        .. code-block:: python

            CheX_Dataset num_samples=191010 views=['PA', 'AP']

        If in a jupyter notebook it will also print the counts of the
        pathology counts returned by .totals()

        .. code-block:: python

            {'Atelectasis': {0.0: 17621, 1.0: 29718},
             'Cardiomegaly': {0.0: 22645, 1.0: 23384},
             'Consolidation': {0.0: 30463, 1.0: 12982},
             ...}

        """

    def check_paths_exist(self):
        if not os.path.isdir(self.imgpath):
            raise Exception("imgpath must be a directory")
        if not os.path.isfile(self.csvpath):
            raise Exception("csvpath must be a file")

    def limit_to_selected_views(self, views):
        """This function is called by subclasses to filter the
        images by view based on the values in .csv['view']
        """
        if type(views) is not list:
            views = [views]
        if '*' in views:
            # if you have the wildcard, the rest are irrelevant
            views = ["*"]
        self.views = views

        # missing data is unknown
        self.csv.view.fillna("UNKNOWN", inplace=True)

        if "*" not in views:
            self.csv = self.csv[self.csv["view"].isin(self.views)]  # Select the view

In [4]:
def normalize(img, maxval, reshape=False):
    """Scales images to be roughly [-1024 1024].

    Call xrv.utils.normalize moving forward.
    """
    return xrv.utils.normalize(img, maxval, reshape)


def apply_transforms(sample, transform, seed=None) -> Dict:
    """Applies transforms to the image and masks.
    The seeds are set so that the transforms that are applied
    to the image are the same that are applied to each mask.
    This way data augmentation will work for segmentation or 
    other tasks which use masks information.
    """

    if seed is None:
        MAX_RAND_VAL = 2147483647
        seed = np.random.randint(MAX_RAND_VAL)

    if transform is not None:
        random.seed(seed)
        torch.random.manual_seed(seed)
        sample["img"] = transform(sample["img"])

        if "pathology_masks" in sample:
            for i in sample["pathology_masks"].keys():
                random.seed(seed)
                torch.random.manual_seed(seed)
                sample["pathology_masks"][i] = transform(sample["pathology_masks"][i])

        if "semantic_masks" in sample:
            for i in sample["semantic_masks"].keys():
                random.seed(seed)
                torch.random.manual_seed(seed)
                sample["semantic_masks"][i] = transform(sample["semantic_masks"][i])

    return sample

In [5]:
class PC_Dataset(Dataset):

    def __init__(self,
                 imgpath,
                 csvpath,
                 avail_imgID,
                 views=["PA"],
                 transform=None,
                 data_aug=None,
                 flat_dir=True,
                 seed=0,
                 unique_patients=True
                 ):

        super(PC_Dataset, self).__init__()
        np.random.seed(seed)  # Reset the seed so all runs are the same.

        self.pathologies = ["Atelectasis", "Consolidation", "Infiltration",
                            "Pneumothorax", "Edema", "Emphysema", "Fibrosis",
                            "Effusion", "Pneumonia", "Pleural_Thickening",
                            "Cardiomegaly", "Nodule", "Mass", "Hernia", "Fracture",
                            "Granuloma", "Flattened Diaphragm", "Bronchiectasis",
                            "Aortic Elongation", "Scoliosis",
                            "Hilar Enlargement", "Tuberculosis",
                            "Air Trapping", "Costophrenic Angle Blunting", "Aortic Atheromatosis",
                            "Hemidiaphragm Elevation",
                            "Support Devices", "Tube'"]  # the Tube' is intentional

        self.pathologies = sorted(self.pathologies)

        mapping = dict()

        mapping["Infiltration"] = ["infiltrates",
                                   "interstitial pattern",
                                   "ground glass pattern",
                                   "reticular interstitial pattern",
                                   "reticulonodular interstitial pattern",
                                   "alveolar pattern",
                                   "consolidation",
                                   "air bronchogram"]
        mapping["Pleural_Thickening"] = ["pleural thickening"]
        mapping["Consolidation"] = ["air bronchogram"]
        mapping["Hilar Enlargement"] = ["adenopathy",
                                        "pulmonary artery enlargement"]
        mapping["Support Devices"] = ["device",
                                      "pacemaker"]
        mapping["Tube'"] = ["stent'"]  # the ' is to select findings which end in that word

        self.imgpath = imgpath
        self.transform = transform
        self.data_aug = data_aug
        self.flat_dir = flat_dir
        self.csvpath = csvpath
        
        self.csv = pd.read_csv(self.csvpath, low_memory=False)

        # Standardize view names
        self.csv.loc[self.csv["Projection"].isin(["AP_horizontal"]), "Projection"] = "AP Supine"

        self.csv["view"] = self.csv['Projection']
        self.limit_to_selected_views(views)
        
        #----------------ADDED------------------
        # Remove null stuff
        #self.csv = self.csv[~self.csv["Labels"].isnull()]
        self.csv['Labels'].fillna('', inplace = True)
        
        # Filter by available image files
        self.csv = self.csv[self.csv["ImageID"].isin(avail_imgID)]
        
        # Exclude the labels:
        self.csv = self.csv[~self.csv["Labels"].str.contains('exclude')]
        self.csv = self.csv[~self.csv["Labels"].str.contains('suboptimal study')]
        #----------------ADDED------------------
        

        #----------------Not used in this case------------------
        if unique_patients:
            self.csv = self.csv.groupby("PatientID").first().reset_index()

        # Filter out age < 10 (paper published 2019)
        #self.csv = self.csv[(2019 - self.csv.PatientBirth > 10)]
        
        #------------------------------------------------------

        # Get our classes.
        labels = []
        for pathology in self.pathologies:
            mask = self.csv["Labels"].str.contains(pathology.lower())
            if pathology in mapping:
                for syn in mapping[pathology]:
                    #print("mapping", syn)
                    mask |= self.csv["Labels"].str.contains(syn.lower())
            labels.append(mask.values)
        self.labels = np.asarray(labels).T
        self.labels = self.labels.astype(np.float32)

        self.pathologies[self.pathologies.index("Tube'")] = "Tube"

        # add consistent csv values

        # offset_day_int
        dt = pd.to_datetime(self.csv["StudyDate_DICOM"], format="%Y%m%d")
        self.csv["offset_day_int"] = dt.astype(np.int64) // 10**9 // 86400

        # patientid
        self.csv["patientid"] = self.csv["PatientID"].astype(str)

        # age
        self.csv['age_years'] = (2017 - self.csv['PatientBirth'])

        # sex
        self.csv['sex_male'] = self.csv['PatientSex_DICOM'] == 'M'
        self.csv['sex_female'] = self.csv['PatientSex_DICOM'] == 'F'

    def string(self):
        return self.__class__.__name__ + " num_samples={} views={} data_aug={}".format(len(self), self.views, self.data_aug)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        sample = {}
        sample["idx"] = idx
        sample["lab"] = self.labels[idx]

        imgid = self.csv['ImageID'].iloc[idx]
        img_path = os.path.join(self.imgpath, imgid)
        img = imread(img_path)

        sample["img"] = normalize(img, maxval=65535, reshape=True)

        sample = apply_transforms(sample, self.transform)
        sample = apply_transforms(sample, self.data_aug)

        return sample

## Define the function to process the data

In [6]:
def process_data(dataset, model):
    outs = []
    labs = []
    with torch.no_grad():
        np.random.seed(0)
        for i in range(len(dataset)):
            sample = dataset[i]
            labs.append(sample["lab"])
            image = torch.from_numpy(sample["img"]).unsqueeze(0)
            if torch.cuda.is_available():
                image = image.cuda()
            out = model(image).cpu()
            outs.append(out.detach().numpy()[0])
    
    return outs, labs

## Define the function to obtain the metrics

In [7]:
def calculate_metrics(index, outs, labs, model):
    result = {}
    result["Pathology"] = model.pathologies[index]
    if len(np.unique(np.asarray(labs)[:, index])) > 1:
        labels = np.asarray(labs)[:, index].astype(bool)
        preds = np.asarray(outs)[:, index]
        result["AUC"] = sklearn.metrics.roc_auc_score(labels, preds)
        result["Acc"] = sklearn.metrics.accuracy_score(labels, preds > 0.5)
        result["Prec"] = sklearn.metrics.precision_score(labels, preds > 0.5)
        result["Recall"] = sklearn.metrics.recall_score(labels, preds > 0.5)
        result["Spec"] = imblearn.metrics.specificity_score(labels, preds > 0.5)
        result["F1"] = sklearn.metrics.f1_score(labels, preds > 0.5)
    return result

## Define the function to show the results for each model/dataset

In [14]:
def model_results(model):
    
    print("Model: ", model)
    # Create an empty dataframe to store the results
    df = pd.DataFrame()
    
    for name in names_to_load:

        # Filter by the corresponding txt file
        imageIDs_available = os.path.join("pathologies_imageIDs", f"{name}.txt")
        with open(imageIDs_available, "r") as f:
            avail_ID = [line.strip() for line in f.readlines()]

        # Load the dataset
        d_pc = PC_Dataset(imgpath = "/home/marialuque/datos_maria/Data/PadChest512_PA_Manually_Filtered",
                     csvpath = "/home/marialuque/datos_maria/Data/PADCHEST_chest_x_ray_images_labels_160K_01.02.19.csv" ,
                     avail_imgID = avail_ID,unique_patients = False  )
        xrv.datasets.relabel_dataset(xrv.datasets.default_pathologies, d_pc, silent = True)

        # Process the dataset and show the results
        if name in name_indices:
            outs, labs = process_data(d_pc, model)
            results = []
            results.append(calculate_metrics(name_indices[name], outs, labs, model))
            df = df.append(results, ignore_index = True)
            
    return df

## Define the list of different pathologies

In [15]:
# List of names to load
names_to_load = ['atelectasis_IDs', 'infiltration_IDs', 'pneumothorax_IDs', 'effusion_IDs', 'pneumonia_IDs', 'cardiomegaly_IDs', 'nodule_IDs','mass_IDs', 'fracture_IDs']

# Define the mapping of names to indices for calculate_metrics()
name_indices = {
    'atelectasis_IDs': 0,
    'infiltration_IDs': 2,
    'pneumothorax_IDs': 3,
    'effusion_IDs': 7,
    'pneumonia_IDs': 8,
    'cardiomegaly_IDs': 10,
    'nodule_IDs': 11,
    'mass_IDs': 12,
    'fracture_IDs': 15
}

## Results for each model

In [20]:
model_list = [model_pc, model_all, model_rsna, model_nih, model_chex, model_mimic_nb, model_mimic_ch]


# Iterate over the model list
for model in model_list:
    df = model_results(model)
    display(df)


Model:  XRV-DenseNet121-densenet121-res224-pc


,Pathology,AUC,Acc,Prec,Recall,Spec,F1
0,Atelectasis,0.789841,0.713597,0.686448,0.786403,0.640792,0.733034
1,Infiltration,0.844847,0.760096,0.739418,0.803279,0.716913,0.770027
2,Pneumothorax,0.901038,0.824885,0.794979,0.875576,0.774194,0.833333
3,Effusion,0.951490,0.886784,0.890513,0.882009,0.891559,0.886241
4,Pneumonia,0.849297,0.770940,0.769717,0.773208,0.768673,0.771459
5,Cardiomegaly,0.931355,0.859872,0.834096,0.898447,0.821297,0.865077
6,Nodule,0.680737,0.623870,0.602191,0.729944,0.517797,0.659941
7,Mass,0.855977,0.783525,0.792876,0.767561,0.799489,0.780013
8,Fracture,0.718549,0.664422,0.646127,0.727021,0.601823,0.684191


Model:  XRV-ResNet-resnet50-res512-all


,Pathology,AUC,Acc,Prec,Recall,Spec,F1
0,Atelectasis,0.792954,0.512633,0.980198,0.025788,0.999479,0.050254
1,Infiltration,0.831990,0.532320,0.958412,0.067573,0.997068,0.126245
2,Pneumothorax,0.826839,0.506912,1.000000,0.013825,1.000000,0.027273
3,Effusion,0.949010,0.657116,0.993230,0.316389,0.997843,0.479907
4,Pneumonia,0.778407,0.514968,0.962617,0.031146,0.998790,0.060340
5,Cardiomegaly,0.931639,0.569448,0.978760,0.141977,0.996919,0.247982
6,Nodule,0.760643,0.520621,0.967949,0.042655,0.998588,0.081710
7,Mass,0.857003,0.559387,0.960396,0.123883,0.994891,0.219457
8,Fracture,0.863488,0.610737,0.981067,0.225832,0.995642,0.367150


Model:  XRV-DenseNet121-densenet121-res224-rsna


,Pathology,AUC,Acc,Prec,Recall,Spec,F1
0,,0.500000,0.500000,0.000000,0.000000,1.000000,0.000000
1,,0.500000,0.500000,0.000000,0.000000,1.000000,0.000000
2,,0.500000,0.500000,0.000000,0.000000,1.000000,0.000000
3,,0.500000,0.500000,0.000000,0.000000,1.000000,0.000000
4,Pneumonia,0.747404,0.673874,0.760653,0.507409,0.840339,0.608743
5,,0.500000,0.500000,0.000000,0.000000,1.000000,0.000000
6,,0.500000,0.500000,0.000000,0.000000,1.000000,0.000000
7,,0.500000,0.500000,0.000000,0.000000,1.000000,0.000000
8,,0.500000,0.500000,0.000000,0.000000,1.000000,0.000000


... (4 outputs left out)

Model:  XRV-DenseNet121-densenet121-res224-mimic_nb


,Pathology,AUC,Acc,Prec,Recall,Spec,F1
0,Atelectasis,0.708005,0.629200,0.592091,0.830685,0.427716,0.691382
1,,0.500000,0.500000,0.000000,0.000000,1.000000,0.000000
2,Pneumothorax,0.700652,0.640553,0.617761,0.737327,0.543779,0.672269
3,Effusion,0.903539,0.805607,0.761880,0.889094,0.722120,0.820586
4,Pneumonia,0.682101,0.574992,0.548934,0.841246,0.308739,0.664358
5,Cardiomegaly,0.855634,0.782783,0.738192,0.876386,0.689179,0.801375
6,,0.500000,0.500000,0.000000,0.000000,1.000000,0.000000
7,,0.500000,0.500000,0.000000,0.000000,1.000000,0.000000
8,Fracture,0.621561,0.573098,0.610149,0.404913,0.741284,0.486783


Model:  XRV-DenseNet121-densenet121-res224-mimic_ch


,Pathology,AUC,Acc,Prec,Recall,Spec,F1
0,Atelectasis,0.715659,0.653556,0.628236,0.752279,0.554832,0.684685
1,,0.500000,0.500000,0.000000,0.000000,1.000000,0.000000
2,Pneumothorax,0.799422,0.629032,0.576503,0.972350,0.285714,0.723842
3,Effusion,0.903940,0.807455,0.764161,0.889402,0.725508,0.822039
4,Pneumonia,0.702844,0.651043,0.635918,0.706683,0.595404,0.669436
5,Cardiomegaly,0.845669,0.736012,0.666754,0.943678,0.528346,0.781406
6,,0.500000,0.500000,0.000000,0.000000,1.000000,0.000000
7,,0.500000,0.500000,0.000000,0.000000,1.000000,0.000000
8,Fracture,0.565166,0.538431,0.581926,0.272979,0.803883,0.371629


In [24]:
dict(zip(model_pc.pathologies,d_pc.pathologies))

{'Atelectasis': 'Atelectasis',
 'Consolidation': 'Consolidation',
 'Infiltration': 'Infiltration',
 'Pneumothorax': 'Pneumothorax',
 'Edema': 'Edema',
 'Emphysema': 'Emphysema',
 'Fibrosis': 'Fibrosis',
 'Effusion': 'Effusion',
 'Pneumonia': 'Pneumonia',
 'Pleural_Thickening': 'Pleural_Thickening',
 'Cardiomegaly': 'Cardiomegaly',
 'Nodule': 'Nodule',
 'Mass': 'Mass',
 'Hernia': 'Hernia',
 '': 'Enlarged Cardiomediastinum',
 'Fracture': 'Fracture'}

In [25]:
dict(zip(model_all.pathologies,d_pc.pathologies))

{'Atelectasis': 'Atelectasis',
 'Consolidation': 'Consolidation',
 'Infiltration': 'Infiltration',
 'Pneumothorax': 'Pneumothorax',
 'Edema': 'Edema',
 'Emphysema': 'Emphysema',
 'Fibrosis': 'Fibrosis',
 'Effusion': 'Effusion',
 'Pneumonia': 'Pneumonia',
 'Pleural_Thickening': 'Pleural_Thickening',
 'Cardiomegaly': 'Cardiomegaly',
 'Nodule': 'Nodule',
 'Mass': 'Mass',
 'Hernia': 'Hernia',
 'Lung Lesion': 'Lung Lesion',
 'Fracture': 'Fracture',
 'Lung Opacity': 'Lung Opacity',
 'Enlarged Cardiomediastinum': 'Enlarged Cardiomediastinum'}

In [26]:
dict(zip(model_rsna.pathologies,d_pc.pathologies))

{'': 'Enlarged Cardiomediastinum',
 'Pneumonia': 'Pneumonia',
 'Lung Opacity': 'Lung Opacity'}

In [28]:
model_rsna.pathologies

['',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 'Pneumonia',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 'Lung Opacity',
 '']

In [27]:
dict(zip(model_nih.pathologies,d_pc.pathologies))

{'Atelectasis': 'Atelectasis',
 'Consolidation': 'Consolidation',
 'Infiltration': 'Infiltration',
 'Pneumothorax': 'Pneumothorax',
 'Edema': 'Edema',
 'Emphysema': 'Emphysema',
 'Fibrosis': 'Fibrosis',
 'Effusion': 'Effusion',
 'Pneumonia': 'Pneumonia',
 'Pleural_Thickening': 'Pleural_Thickening',
 'Cardiomegaly': 'Cardiomegaly',
 'Nodule': 'Nodule',
 'Mass': 'Mass',
 'Hernia': 'Hernia',
 '': 'Enlarged Cardiomediastinum'}

In [56]:
dict(zip(model_chex.pathologies,d_pc.pathologies))

{'Atelectasis': 'Atelectasis',
 'Consolidation': 'Consolidation',
 '': 'Hernia',
 'Pneumothorax': 'Pneumothorax',
 'Edema': 'Edema',
 'Effusion': 'Effusion',
 'Pneumonia': 'Pneumonia',
 'Cardiomegaly': 'Cardiomegaly',
 'Lung Lesion': 'Lung Lesion',
 'Fracture': 'Fracture',
 'Lung Opacity': 'Lung Opacity',
 'Enlarged Cardiomediastinum': 'Enlarged Cardiomediastinum'}

In [57]:
dict(zip(model_mimic_nb.pathologies,d_pc.pathologies))

{'Atelectasis': 'Atelectasis',
 'Consolidation': 'Consolidation',
 '': 'Hernia',
 'Pneumothorax': 'Pneumothorax',
 'Edema': 'Edema',
 'Effusion': 'Effusion',
 'Pneumonia': 'Pneumonia',
 'Cardiomegaly': 'Cardiomegaly',
 'Lung Lesion': 'Lung Lesion',
 'Fracture': 'Fracture',
 'Lung Opacity': 'Lung Opacity',
 'Enlarged Cardiomediastinum': 'Enlarged Cardiomediastinum'}

In [58]:
dict(zip(model_mimic_ch.pathologies,d_pc.pathologies))

{'Atelectasis': 'Atelectasis',
 'Consolidation': 'Consolidation',
 '': 'Hernia',
 'Pneumothorax': 'Pneumothorax',
 'Edema': 'Edema',
 'Effusion': 'Effusion',
 'Pneumonia': 'Pneumonia',
 'Cardiomegaly': 'Cardiomegaly',
 'Lung Lesion': 'Lung Lesion',
 'Fracture': 'Fracture',
 'Lung Opacity': 'Lung Opacity',
 'Enlarged Cardiomediastinum': 'Enlarged Cardiomediastinum'}